# Notebook 06 — Evil Model: Synthetic Borderline Object Generation

**Project:** Planetary Defence Officer — ESA NEOCC Simulator  
**Inputs:** `output/dataset_full.npz`, `output/feature_names.json`, `output/scaler.pkl`, `output/rf_full.pkl`  
**Outputs:** `output/synthetic_raw.json`, `output/synthetic_validated.json`, `output/synthetic_pool.json`, `output/synthetic_validation_pairs.png`, `output/synthetic_attrition.json`

**Purpose:** Generate ~750 physically-valid synthetic borderline-hazardous asteroids for the
game pool. These objects must pass a four-step physical validation pipeline and must cluster
in the 0.45–0.75 RF confidence band — the borderline zone where the classifier is genuinely
uncertain. These are the objects that create gameplay tension.

**Critical design — independent vs derived features:**

SMOTE is applied **only** to independent physical parameters:
`a, e, i, MOID, H, diameter_min, rel_v, albedo`

Derived parameters are computed **after** SMOTE from physical formulas:
`q = a(1−e)`, `Q = a(1+e)`, `diameter_max = 1329 × 10^(−H/5) / √albedo`

If SMOTE is applied to all features together, `q` and `Q` drift out of consistency with
`a` and `e` — the resulting "asteroids" violate Kepler geometry and are mathematical nonsense.

**Expected attrition: 30–50%.** This is a portfolio metric, not a failure.

Random state: `42` everywhere.

---

## 1. Load artifacts and set up feature map

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, pickle, os
from scipy.spatial import Delaunay

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from imblearn.over_sampling import BorderlineSMOTE

os.makedirs('output', exist_ok=True)

PALETTE = {
    'bg': '#080808', 'surface': '#111111', 'border': '#363636',
    'muted': '#707070', 'text': '#c0c0c0', 'bright': '#dedede',
    'ok': '#6aac79', 'warn': '#c8a235', 'crit': '#c84040',
}
plt.rcParams.update({
    'figure.facecolor': PALETTE['bg'], 'axes.facecolor': PALETTE['bg'],
    'axes.edgecolor': PALETTE['border'], 'text.color': PALETTE['text'],
    'axes.labelcolor': PALETTE['text'], 'xtick.color': PALETTE['muted'],
    'ytick.color': PALETTE['muted'], 'grid.color': '#1a1a1a',
    'grid.alpha': 0.6, 'font.family': 'monospace',
    'axes.titlecolor': PALETTE['bright'], 'axes.titlesize': 11,
    'legend.facecolor': '#080808', 'legend.edgecolor': '#363636',
    'legend.labelcolor': '#c0c0c0',
})

data = np.load('output/dataset_full.npz', allow_pickle=True)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

with open('output/feature_names.json') as f:
    feature_names = json.load(f)

with open('output/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('output/rf_full.pkl', 'rb') as f:
    rf_full = pickle.load(f)

fn = {name: idx for idx, name in enumerate(feature_names)}
print("Feature → index map:")
for name, idx in fn.items():
    print(f"  {idx:2d}  {name}")

## 2. Recover original units and compute albedo

We inverse-transform the scaled training data to work in physical units.
Albedo is not in the dataset directly — it is estimated from the standard
absolute-magnitude / diameter relationship:

`D_km = (1329 / √albedo) × 10^(−H/5)`  →  `albedo = (1329 × 10^(−H/5) / D_km)²`

Albedo is then included as an independent feature for SMOTE, and used after SMOTE
to reconstruct `diameter_max` for each synthetic candidate.

In [ ]:
# Inverse transform to original units
X_train_orig = scaler.inverse_transform(X_train)
X_test_orig  = scaler.inverse_transform(X_test)

# Extract columns
H_tr      = X_train_orig[:, fn['Absolute Magnitude']]
dmin_tr   = X_train_orig[:, fn['Est Dia in KM(min)']]
dmax_tr   = X_train_orig[:, fn['Est Dia in KM(max)']]
relv_tr   = X_train_orig[:, fn['Relative Velocity km per sec']]
moid_tr   = X_train_orig[:, fn['Minimum Orbit Intersection']]
e_tr      = X_train_orig[:, fn['Eccentricity']]
a_tr      = X_train_orig[:, fn['Semi Major Axis']]
i_tr      = X_train_orig[:, fn['Inclination']]
miss_tr   = X_train_orig[:, fn['Miss Dist.(Astronomical)']]
unc_tr    = X_train_orig[:, fn['Orbit Uncertainity']]

# Compute albedo from H and diameter_max
# Use geometric mean of min/max as effective diameter
D_eff_tr  = np.sqrt(np.maximum(dmin_tr * dmax_tr, 1e-9))
albedo_tr = np.clip((1329.0 * 10.0**(-H_tr / 5.0) / D_eff_tr)**2, 1e-4, 0.99)

print(f"Albedo range in training set:  min={albedo_tr.min():.4f}  max={albedo_tr.max():.4f}  mean={albedo_tr.mean():.4f}")
print(f"Among hazardous objects:  min={albedo_tr[y_train==1].min():.4f}  max={albedo_tr[y_train==1].max():.4f}  mean={albedo_tr[y_train==1].mean():.4f}")

# ── Build 8D independent feature matrix ──────────────────────────────────────
# Column order: [H, dmin, relv, MOID, e, a, i, albedo]
IND_COLS = ['H', 'diam_min', 'rel_v', 'MOID', 'e', 'a', 'i', 'albedo']
X8_train = np.column_stack([H_tr, dmin_tr, relv_tr, moid_tr, e_tr, a_tr, i_tr, albedo_tr])

# Scale the 8D space for SMOTE
scaler_8d = StandardScaler()
X8_train_sc = scaler_8d.fit_transform(X8_train)

n_haz   = y_train.sum()
n_safe  = (y_train == 0).sum()
print(f"\n8D training set: {X8_train.shape}  |  hazardous: {n_haz}  safe: {n_safe}")

## 3. Borderline-SMOTE on 8D independent feature space

We generate ~2500 synthetic candidates — substantially more than the 750 needed —
to absorb the expected 30–50% validation attrition across the four steps below.

`sampling_strategy=1.0` balances the classes, giving us ≈ n_safe − n_haz = 2541 synthetic
hazardous samples. All synthetic rows returned by SMOTE are collected as candidates.

In [ ]:
print("Running Borderline-SMOTE on 8D independent feature space...")
smote = BorderlineSMOTE(random_state=42, kind='borderline-1', sampling_strategy=1.0)
X8_res_sc, y_res = smote.fit_resample(X8_train_sc, y_train)

# Synthetic rows are everything added beyond the original training set
n_orig   = len(X8_train_sc)
X8_syn_sc = X8_res_sc[n_orig:]
n_raw    = len(X8_syn_sc)
print(f"Training rows before SMOTE : {n_orig:,}  (hazardous: {n_haz:,})")
print(f"Synthetic candidates generated: {n_raw:,}")

# Inverse-transform to get original physical units
X8_syn = scaler_8d.inverse_transform(X8_syn_sc)
H_syn, dmin_syn, relv_syn, moid_syn, e_syn, a_syn, i_syn, albedo_syn = X8_syn.T
print(f"\nRaw synthetic ranges:")
print(f"  a    : {a_syn.min():.3f} – {a_syn.max():.3f} AU")
print(f"  e    : {e_syn.min():.3f} – {e_syn.max():.3f}")
print(f"  MOID : {moid_syn.min():.4f} – {moid_syn.max():.4f} AU")
print(f"  H    : {H_syn.min():.1f} – {H_syn.max():.1f}")

## 4. Compute derived features

After SMOTE we reconstruct all 13 features needed for RF evaluation:
- `q = a(1−e)`, `Q = a(1+e)` — from Kepler geometry
- `diameter_max = 1329 × 10^(−H/5) / √albedo` — brightness / size relationship
- `TJ = a_J/a + 2 cos(i) √(a(1−e²)/a_J)` — Jupiter Tisserand invariant (a_J = 5.2 AU)
- `miss_dist` — approximated as MOID + small positive jitter (miss distance ≥ MOID by definition)
- `orbit_unc` — set to 0 (synthetic objects have no observational uncertainty)

In [ ]:
A_JUP = 5.2  # Jupiter semi-major axis in AU

q_syn      = a_syn * (1.0 - e_syn)
Q_syn      = a_syn * (1.0 + e_syn)
dmax_syn   = 1329.0 * 10.0**(-H_syn / 5.0) / np.sqrt(np.maximum(albedo_syn, 1e-9))
i_rad      = np.radians(i_syn)
TJ_syn     = A_JUP / a_syn + 2.0 * np.cos(i_rad) * np.sqrt(a_syn * (1.0 - e_syn**2) / A_JUP)

rng = np.random.default_rng(42)
miss_syn   = moid_syn + rng.uniform(0, 0.01, size=n_raw)
unc_syn    = np.zeros(n_raw)

# Save raw synthetic output before validation
raw_records = []
for k in range(n_raw):
    raw_records.append({
        'H': float(H_syn[k]),         'diameter_min': float(dmin_syn[k]),
        'diameter_max': float(dmax_syn[k]), 'rel_v': float(relv_syn[k]),
        'miss_dist_au': float(miss_syn[k]), 'orbit_unc': float(unc_syn[k]),
        'moid': float(moid_syn[k]),   'TJ': float(TJ_syn[k]),
        'e': float(e_syn[k]),          'a': float(a_syn[k]),
        'i': float(i_syn[k]),          'q': float(q_syn[k]),
        'Q': float(Q_syn[k]),          'albedo': float(albedo_syn[k]),
    })

with open('output/synthetic_raw.json', 'w') as f:
    json.dump(raw_records, f)
print(f"Saved {n_raw:,} raw synthetic records → output/synthetic_raw.json")

## 5. Validation Step 1 — Hard physical clamps

Drop any candidate that violates known physical bounds for NEOs.
**Violators are dropped, not clipped** — clipping would pile objects at boundary values,
creating an artificial edge-spike artefact in the feature distributions.

In [ ]:
# Physical validity masks — all must be True to survive
mask_e    = (e_syn >= 0.0) & (e_syn < 0.99)      # avoid hyperbolic/parabolic
mask_i    = (i_syn >= 0.0) & (i_syn <= 180.0)
mask_a    = a_syn > 0.0
mask_dmin = dmin_syn > 0.0
mask_dmax = dmax_syn > 0.0
mask_moid = moid_syn >= 0.0
mask_relv = (relv_syn >= 5.0) & (relv_syn <= 40.0)   # realistic NEO velocity range
mask_H    = (H_syn >= 10.0) & (H_syn <= 33.0)         # astronomically meaningful magnitude
mask_alb  = (albedo_syn > 0.0) & (albedo_syn <= 1.0)
mask_q    = q_syn > 0.0
mask_TJ   = np.isfinite(TJ_syn)

clamp_mask = (mask_e & mask_i & mask_a & mask_dmin & mask_dmax &
              mask_moid & mask_relv & mask_H & mask_alb & mask_q & mask_TJ)

n_after_clamp = clamp_mask.sum()
n_dropped_clamp = n_raw - n_after_clamp
print(f"Before clamps : {n_raw:,}")
print(f"After  clamps : {n_after_clamp:,}  (dropped {n_dropped_clamp:,}  = {100*n_dropped_clamp/n_raw:.1f}%)")

# Apply mask to all arrays
H_c, dmin_c, dmax_c = H_syn[clamp_mask], dmin_syn[clamp_mask], dmax_syn[clamp_mask]
relv_c, miss_c, unc_c = relv_syn[clamp_mask], miss_syn[clamp_mask], unc_syn[clamp_mask]
moid_c, TJ_c = moid_syn[clamp_mask], TJ_syn[clamp_mask]
e_c, a_c, i_c = e_syn[clamp_mask], a_syn[clamp_mask], i_syn[clamp_mask]
q_c, Q_c, alb_c = q_syn[clamp_mask], Q_syn[clamp_mask], albedo_syn[clamp_mask]

## 6. Validation Step 2 — Nearest-neighbour proximity check

`scipy.spatial.Delaunay` degenerates in 8D on this dataset: MOID is near-constant
across all real PHAs (all < 0.05 AU by definition), making the feature space nearly
singular for Qhull.

We use a **KNN distance threshold** instead — more robust in high dimensions and more
interpretable: each synthetic candidate must lie within the 95th percentile of the
pairwise nearest-neighbour distances among real PHAs in the scaled 8D space.
This keeps synthetic points within the natural spread of the real training distribution.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Real hazardous training objects in 8D scaled space
X8_haz_real_sc = X8_train_sc[y_train == 1]   # shape (n_haz, 8)

# Fit KNN on real PHAs
knn = NearestNeighbors(n_neighbors=6, metric='euclidean')
knn.fit(X8_haz_real_sc)

# 95th-percentile pairwise distance among real PHAs (k=5, excluding self)
real_dists, _ = knn.kneighbors(X8_haz_real_sc)
threshold_dist = np.percentile(real_dists[:, 1:], 95)  # exclude self (col 0)
print(f"KNN threshold (95th pct real PHA pairwise dist): {threshold_dist:.4f}")

# Synthetic candidates in 8D scaled space (after clamp)
X8_clamped_sc = scaler_8d.transform(
    np.column_stack([H_c, dmin_c, relv_c, moid_c, e_c, a_c, i_c, alb_c])
)

syn_dists, _ = knn.kneighbors(X8_clamped_sc, n_neighbors=1)
hull_mask = syn_dists[:, 0] <= threshold_dist

n_after_hull  = hull_mask.sum()
n_dropped_hull = n_after_clamp - n_after_hull
print(f"Before proximity check : {n_after_clamp:,}")
print(f"After  proximity check : {n_after_hull:,}  (dropped {n_dropped_hull:,}  = {100*n_dropped_hull/n_after_clamp:.1f}% of clamped)")

H_h, dmin_h, dmax_h = H_c[hull_mask], dmin_c[hull_mask], dmax_c[hull_mask]
relv_h, miss_h, unc_h = relv_c[hull_mask], miss_c[hull_mask], unc_c[hull_mask]
moid_h, TJ_h = moid_c[hull_mask], TJ_c[hull_mask]
e_h, a_h, i_h = e_c[hull_mask], a_c[hull_mask], i_c[hull_mask]
q_h, Q_h, alb_h = q_c[hull_mask], Q_c[hull_mask], alb_c[hull_mask]

## 7. Validation Step 3 — RF confidence filter (borderline zone)

We run `rf_full` on each surviving candidate (assembled into the full 13-feature
scaled space) and keep only those where the predicted hazardous probability lies in
`[0.45, 0.75]` — the zone where the classifier is genuinely uncertain.

This is what makes these objects "borderline" in the meaningful sense: the model
trained on all real data cannot confidently classify them. That uncertainty is
what translates into gameplay difficulty.

In [ ]:
# Assemble 13-feature matrix in the original scaler's expected column order
# Order from feature_names.json:
# [H, dmin, dmax, relv, miss, unc, MOID, TJ, e, a, i, q, Q]
X13_h = np.column_stack([
    H_h, dmin_h, dmax_h, relv_h, miss_h, unc_h,
    moid_h, TJ_h, e_h, a_h, i_h, q_h, Q_h,
])

# Scale with original 13-feature scaler
X13_h_sc = scaler.transform(X13_h)

# RF confidence (hazardous probability)
conf_h = rf_full.predict_proba(X13_h_sc)[:, 1]

# Keep borderline zone: 0.45 ≤ confidence ≤ 0.75
conf_mask = (conf_h >= 0.45) & (conf_h <= 0.75)
n_after_conf = conf_mask.sum()
n_dropped_conf = n_after_hull - n_after_conf

print(f"Before confidence filter : {n_after_hull:,}")
print(f"After  confidence filter : {n_after_conf:,}  (dropped {n_dropped_conf:,}  = {100*n_dropped_conf/n_after_hull:.1f}% of hull-validated)")
print(f"\nConfidence distribution of hull-validated candidates:")
print(f"  < 0.45  (clearly safe)      : {(conf_h < 0.45).sum():,}")
print(f"  0.45–0.75 (borderline zone) : {n_after_conf:,}")
print(f"  > 0.75  (clearly hazardous) : {(conf_h > 0.75).sum():,}")

H_v       = H_h[conf_mask];    dmin_v   = dmin_h[conf_mask]
dmax_v    = dmax_h[conf_mask]; relv_v   = relv_h[conf_mask]
miss_v    = miss_h[conf_mask]; unc_v    = unc_h[conf_mask]
moid_v    = moid_h[conf_mask]; TJ_v     = TJ_h[conf_mask]
e_v       = e_h[conf_mask];    a_v      = a_h[conf_mask]
i_v       = i_h[conf_mask];    q_v      = q_h[conf_mask]
Q_v       = Q_h[conf_mask];    alb_v    = alb_h[conf_mask]
conf_v    = conf_h[conf_mask]

print(f"\nBorderline confidence stats:  mean={conf_v.mean():.3f}  std={conf_v.std():.3f}  min={conf_v.min():.3f}  max={conf_v.max():.3f}")

## 8. Validation Step 4 — Visual pair plot sanity check

Real PHAs and validated synthetic objects plotted side-by-side on key feature axes.
**If synthetic forms a visually separate cluster from real PHAs, stop and debug.**
We expect synthetic to lie *among* real objects, not in a detached cloud.

In [ ]:
# Real PHA training objects in original units
X8_haz_real_orig = X8_train[y_train == 1]
H_real    = X8_haz_real_orig[:, 0]
dmin_real = X8_haz_real_orig[:, 1]
moid_real = X8_haz_real_orig[:, 3]
e_real    = X8_haz_real_orig[:, 4]
a_real    = X8_haz_real_orig[:, 5]
dmax_real = X_train_orig[y_train == 1, fn['Est Dia in KM(max)']]

PAIR_FEATURES = {
    'Semi-Major a (AU)':  (a_real,    a_v),
    'Eccentricity e':     (e_real,    e_v),
    'MOID (AU)':          (moid_real, moid_v),
    'Diam. max (km)':     (dmax_real, dmax_v),
}
labels = list(PAIR_FEATURES.keys())
vals   = list(PAIR_FEATURES.values())
n_feat = len(labels)

fig, axes = plt.subplots(n_feat, n_feat, figsize=(12, 12))
fig.patch.set_facecolor(PALETTE['bg'])

for r in range(n_feat):
    for c in range(n_feat):
        ax = axes[r, c]
        ax.set_facecolor(PALETTE['bg'])
        for spine in ax.spines.values():
            spine.set_edgecolor(PALETTE['border'])

        r_real, r_syn = vals[r]
        c_real, c_syn = vals[c]

        p99_r = np.percentile(np.concatenate([r_real, r_syn]), 99)
        p99_c = np.percentile(np.concatenate([c_real, c_syn]), 99)

        if r == c:
            ax.hist(r_real, bins=25, alpha=0.6, color=PALETTE['ok'],
                    density=True, label='Real PHA')
            ax.hist(r_syn,  bins=25, alpha=0.6, color=PALETTE['crit'],
                    density=True, label='Synthetic')
            if r == 0:
                ax.legend(fontsize=7, loc='upper right')
        else:
            ax.scatter(c_real, r_real, s=8, alpha=0.5, color=PALETTE['ok'],   label='Real PHA')
            ax.scatter(c_syn,  r_syn,  s=8, alpha=0.5, color=PALETTE['crit'], label='Synthetic')
            ax.set_xlim(-0.02 * p99_c, 1.05 * p99_c)
            ax.set_ylim(-0.02 * p99_r, 1.05 * p99_r)

        if r == n_feat - 1:
            ax.set_xlabel(labels[c], fontsize=8)
        else:
            ax.set_xticklabels([])
        if c == 0:
            ax.set_ylabel(labels[r], fontsize=8)
        else:
            ax.set_yticklabels([])
        ax.tick_params(labelsize=7)

fig.suptitle('◈ VALIDATION STEP 4 — Real PHA (green) vs Synthetic Borderline (red)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('output/synthetic_validation_pairs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved output/synthetic_validation_pairs.png")
print()
print("VISUAL CHECK: Synthetic objects should lie *among* real PHAs, not in a separate cloud.")
print("If they form a detached cluster → stop and debug before proceeding.")

## 9. Select final 750 and save pool

In [ ]:
if n_after_conf < 750:
    print(f"WARNING: only {n_after_conf} validated borderline objects available.")
    print("Using all of them. Consider lowering confidence threshold or generating more candidates.")
    n_select = n_after_conf
else:
    n_select = 750

rng_sel = np.random.default_rng(42)
sel_idx = rng_sel.choice(n_after_conf, size=n_select, replace=False)

pool_records = []
for k in sel_idx:
    pool_records.append({
        'H':            float(H_v[k]),
        'diameter_min': float(dmin_v[k]),
        'diameter_max': float(dmax_v[k]),
        'rel_v':        float(relv_v[k]),
        'miss_dist_au': float(miss_v[k]),
        'orbit_unc':    float(unc_v[k]),
        'moid':         float(moid_v[k]),
        'TJ':           float(TJ_v[k]),
        'e':            float(e_v[k]),
        'a':            float(a_v[k]),
        'i':            float(i_v[k]),
        'q':            float(q_v[k]),
        'Q':            float(Q_v[k]),
        'albedo':       float(alb_v[k]),
        'model_confidence': float(conf_v[k]),
        'model_prediction': 'hazardous',
        'true_class':   'hazardous',
        'is_synthetic': True,
    })

# Validated (pre-selection) pool
validated_records = []
for k in range(n_after_conf):
    validated_records.append({
        'H': float(H_v[k]), 'diameter_min': float(dmin_v[k]),
        'diameter_max': float(dmax_v[k]), 'rel_v': float(relv_v[k]),
        'miss_dist_au': float(miss_v[k]), 'orbit_unc': float(unc_v[k]),
        'moid': float(moid_v[k]), 'TJ': float(TJ_v[k]),
        'e': float(e_v[k]), 'a': float(a_v[k]), 'i': float(i_v[k]),
        'q': float(q_v[k]), 'Q': float(Q_v[k]), 'albedo': float(alb_v[k]),
        'model_confidence': float(conf_v[k]),
    })

with open('output/synthetic_validated.json', 'w') as f:
    json.dump(validated_records, f)
with open('output/synthetic_pool.json', 'w') as f:
    json.dump(pool_records, f)

print(f"✓ output/synthetic_validated.json  ({len(validated_records)} objects)")
print(f"✓ output/synthetic_pool.json       ({len(pool_records)} objects — final game pool)")

## 10. Attrition summary

In [ ]:
attrition = {
    'raw_smote':              int(n_raw),
    'after_physical_clamps':  int(n_after_clamp),
    'after_convex_hull':      int(n_after_hull),
    'after_confidence_filter':int(n_after_conf),
    'final_pool':             int(n_select),
    'attrition_rates': {
        'clamp_drop_pct':   round(100 * (n_raw - n_after_clamp) / n_raw, 1),
        'hull_drop_pct':    round(100 * (n_after_clamp - n_after_hull) / n_after_clamp, 1),
        'conf_drop_pct':    round(100 * (n_after_hull - n_after_conf) / n_after_hull, 1),
        'total_attrition':  round(100 * (n_raw - n_after_conf) / n_raw, 1),
    },
    'borderline_confidence': {
        'mean': round(float(conf_v.mean()), 4),
        'std':  round(float(conf_v.std()),  4),
        'min':  round(float(conf_v.min()),  4),
        'max':  round(float(conf_v.max()),  4),
    },
    'final_pool_ranges': {
        'moid_min': round(float(moid_v[sel_idx].min()), 4),
        'moid_max': round(float(moid_v[sel_idx].max()), 4),
        'H_min':    round(float(H_v[sel_idx].min()),    2),
        'H_max':    round(float(H_v[sel_idx].max()),    2),
    }
}

with open('output/synthetic_attrition.json', 'w') as f:
    json.dump(attrition, f, indent=2)

print("◈ SYNTHETIC GENERATION ATTRITION SUMMARY")
print("=" * 55)
print(f"  Raw SMOTE candidates          : {attrition['raw_smote']:>6,}")
print(f"  After physical clamps         : {attrition['after_physical_clamps']:>6,}  ({attrition['attrition_rates']['clamp_drop_pct']:.1f}% dropped)")
print(f"  After convex hull             : {attrition['after_convex_hull']:>6,}  ({attrition['attrition_rates']['hull_drop_pct']:.1f}% dropped)")
print(f"  After confidence [0.45,0.75]  : {attrition['after_confidence_filter']:>6,}  ({attrition['attrition_rates']['conf_drop_pct']:.1f}% dropped)")
print(f"  ─────────────────────────────────────────────")
print(f"  Total attrition               :         {attrition['attrition_rates']['total_attrition']:.1f}%")
print(f"  Selected for game pool        : {attrition['final_pool']:>6,}")
print(f"\nBorderline confidence — mean={conf_v.mean():.3f}  std={conf_v.std():.3f}")
print(f"MOID range in pool            — {attrition['final_pool_ranges']['moid_min']:.4f} – {attrition['final_pool_ranges']['moid_max']:.4f} AU")
print(f"H range in pool               — {attrition['final_pool_ranges']['H_min']:.1f} – {attrition['final_pool_ranges']['H_max']:.1f}")
print(f"\n✓ output/synthetic_attrition.json saved")
print(f"\nPhase 5 complete. {n_select} synthetic borderline objects ready for export.")

## Summary

The four-step physical validation pipeline filters raw SMOTE output down to a set of
synthetic PHAs that are:

1. **Physically plausible** — pass hard orbital/physical bounds (clamp step)
2. **Within the real NEO distribution** — inside the convex hull of real PHA training
   points in feature space (hull step)
3. **Genuinely borderline** — the trained classifier cannot confidently categorise them;
   they sit in the 0.45–0.75 confidence band (confidence step)
4. **Visually indistinguishable from real PHAs** — pair plot shows synthetic objects
   distributed among real ones, not in a separate cloud (visual step)

The attrition rate (~30–50% expected) is itself a methodological result: it demonstrates
that Borderline-SMOTE does not generate "free" hard training examples. A meaningful
fraction of synthetic points are physically invalid or extrapolate beyond the training
distribution, and the confidence filter ensures only the genuinely uncertain objects
reach the game pool.

These objects are the core gameplay resource for difficulty levels 2 and 3. Real PHAs
in the dataset are classified with near-certainty by the RF model (MOID clearly below
0.05 AU) — they make good tutorial objects but not good challenge objects. The synthetic
borderline pool is what makes the game hard.